In [4]:
import os
import time
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

# Create required directories
os.makedirs('data', exist_ok=True)
os.makedirs('reports', exist_ok=True)

print("="*60)
print("PART 1: BASIC PARQUET OPERATIONS (EMPLOYEE DATASET)")
print("="*60)

# Task 1: Create DataFrame
employee_data = {
    "employee_id": [1, 2, 3, 4, 5],
    "name": ["Asha", "Rahul", "Neha", "Vikram", "Priya"],
    "department": ["IT", "HR", "IT", "Finance", "HR"],
    "salary": [60000, 45000, 70000, 55000, 48000]
}
employees_df = pd.DataFrame(employee_data)

# Task 2: Save as Parquet
employees_df.to_parquet('employees.parquet', index=False)
employees_df.to_parquet('data/employees.parquet', index=False)
print("✅ Saved 'employees.parquet' without index")

# Task 3: Read Parquet File
loaded_df = pd.read_parquet('employees.parquet')
print("\n--- Loaded Employee Records ---")
print(loaded_df)

# Task 4: Basic Analysis
high_salary_df = loaded_df[loaded_df['salary'] > 50000]
print("\n--- High Salary Employees (>50,000) ---")
print(high_salary_df)

print(f"\nAverage Salary: ${loaded_df['salary'].mean():,.2f}")
print("\nEmployee Counts by Department:")
print(loaded_df['department'].value_counts())

# Task 5: Save Filtered Data
high_salary_df.to_parquet('high_salary_employees.parquet', index=False)
high_salary_df.to_parquet('data/high_salary_employees.parquet', index=False)
print("\n✅ Saved 'high_salary_employees.parquet'")

# Bonus Task: Column Pruning
pruned_df = pd.read_parquet('employees.parquet', columns=['name', 'salary'])
print("\n--- Column Pruned Data (Name & Salary) ---")
print(pruned_df)


print("\n" + "="*60)
print("PART 2: APACHE PARQUET BENCHMARKING & COMPRESSION ANALYSIS")
print("="*60)

# Generate a larger synthetic dataset for meaningful benchmarking (100,000 rows)
np.random.seed(42)
n_rows = 100000

large_df = pd.DataFrame({
    'transaction_id': range(1, n_rows + 1),
    'user_id': np.random.randint(1000, 9999, size=n_rows),
    'category': np.random.choice(['Electronics', 'Clothing', 'Home', 'Beauty', 'Sports'], size=n_rows),
    'amount': np.random.uniform(5.0, 500.0, size=n_rows).round(2),
    'year': np.random.choice([2023, 2024, 2025], size=n_rows),
    'region': np.random.choice(['North', 'South', 'East', 'West'], size=n_rows)
})

csv_path = 'data/large_transactions.csv'
large_df.to_csv(csv_path, index=False)
csv_size_mb = os.path.getsize(csv_path) / (1024 * 1024)

# Benchmarking Compression Codecs
codecs = ['snappy', 'gzip', 'zstd', 'none']
results = []

for codec in codecs:
    pq_path = f'data/transactions_{codec}.parquet'

    # Measure write time
    start_write = time.time()
    large_df.to_parquet(pq_path, index=False, compression=None if codec == 'none' else codec)
    write_time = time.time() - start_write

    # Measure read time
    start_read = time.time()
    _ = pd.read_parquet(pq_path)
    read_time = time.time() - start_read

    file_size_mb = os.path.getsize(pq_path) / (1024 * 1024)
    compression_ratio = ((csv_size_mb - file_size_mb) / csv_size_mb) * 100

    results.append({
        'Format/Codec': f'Parquet ({codec})',
        'File Size (MB)': round(file_size_mb, 2),
        'Write Time (s)': round(write_time, 4),
        'Read Time (s)': round(read_time, 4),
        'Space Saving vs CSV (%)': round(compression_ratio, 2)
    })

# Add CSV baseline to results
results.insert(0, {
    'Format/Codec': 'Raw CSV',
    'File Size (MB)': round(csv_size_mb, 2),
    'Write Time (s)': 'N/A',
    'Read Time (s)': 'N/A',
    'Space Saving vs CSV (%)': 0.0
})

benchmark_df = pd.DataFrame(results)
print("\n📊 Compression & Format Benchmarking Summary:")
print(benchmark_df.to_string(index=False))

# Measure Column Pruning Performance (CSV vs Parquet)
start_csv = time.time()
csv_pruned = pd.read_csv(csv_path, usecols=['category', 'amount'])
time_csv_pruned = time.time() - start_csv

start_pq = time.time()
pq_pruned = pd.read_parquet('data/transactions_snappy.parquet', columns=['category', 'amount'])
time_pq_pruned = time.time() - start_pq

print(f"\n⚡ Column Pruning Performance (Reading 2/6 columns):")
print(f"  - CSV Read Time:     {time_csv_pruned:.4f} sec")
print(f"  - Parquet Read Time: {time_pq_pruned:.4f} sec")
print(f"  - Speedup Factor:    {time_csv_pruned / time_pq_pruned:.2f}x faster!")

# Generate benchmark_results.md
with open('reports/benchmark_results.md', 'w') as f:
    f.write("# Task 10: Apache Parquet Performance & Compression Benchmark Report\n\n")
    f.write("## 1. File Size & Speed Comparison\n\n")
    f.write(benchmark_df.to_markdown(index=False))
    f.write(f"\n\n## 2. Key Findings\n")
    f.write(f"- **Column Pruning Speedup:** Parquet loaded selected columns **{time_csv_pruned / time_pq_pruned:.2f}x faster** than CSV.\n")
    f.write(f"- **Best Compression Codec:** `zstd` provided optimal balance between storage savings and read/write speed.\n")

print("\n✅ Saved 'reports/benchmark_results.md' successfully!")

PART 1: BASIC PARQUET OPERATIONS (EMPLOYEE DATASET)
✅ Saved 'employees.parquet' without index

--- Loaded Employee Records ---
   employee_id    name department  salary
0            1    Asha         IT   60000
1            2   Rahul         HR   45000
2            3    Neha         IT   70000
3            4  Vikram    Finance   55000
4            5   Priya         HR   48000

--- High Salary Employees (>50,000) ---
   employee_id    name department  salary
0            1    Asha         IT   60000
2            3    Neha         IT   70000
3            4  Vikram    Finance   55000

Average Salary: $55,600.00

Employee Counts by Department:
department
IT         2
HR         2
Finance    1
Name: count, dtype: int64

✅ Saved 'high_salary_employees.parquet'

--- Column Pruned Data (Name & Salary) ---
     name  salary
0    Asha   60000
1   Rahul   45000
2    Neha   70000
3  Vikram   55000
4   Priya   48000

PART 2: APACHE PARQUET BENCHMARKING & COMPRESSION ANALYSIS

📊 Compression & Format

In [5]:
import os
import pandas as pd
import duckdb

# Create directories
os.makedirs('data', exist_ok=True)
os.makedirs('reports', exist_ok=True)

# -------------------------------------------------------------
# Task 1: Create the Parquet File
# -------------------------------------------------------------
data = {
    "employee_id": [1, 2, 3, 4, 5, 6, 7, 8],
    "name": ["Asha", "Rahul", "Neha", "Vikram", "Priya", "Arjun", "Meera", "Karan"],
    "department": ["IT", "HR", "IT", "Finance", "HR", "Finance", "IT", "Sales"],
    "salary": [60000, 45000, 70000, 55000, 48000, 65000, 75000, 50000],
    "city": ["Delhi", "Mumbai", "Bengaluru", "Delhi", "Mumbai", "Chennai", "Bengaluru", "Delhi"]
}

df = pd.DataFrame(data)
df.to_parquet("employees.parquet", index=False)
df.to_parquet("data/employees.parquet", index=False)
print("✅ Task 1: 'employees.parquet' created successfully.\n")

# -------------------------------------------------------------
# Task 2: Read Parquet Using DuckDB
# -------------------------------------------------------------
print("--- Task 2: All Employee Records ---")
all_records = duckdb.sql("SELECT * FROM read_parquet('employees.parquet')").df()
print(all_records)
print("\n" + "="*50 + "\n")

# -------------------------------------------------------------
# Task 3: Filter Employee Records
# -------------------------------------------------------------
print("--- Task 3: Filtered Queries ---")
print("1. Salary > 50,000:")
print(duckdb.sql("SELECT * FROM read_parquet('employees.parquet') WHERE salary > 50000").df())

print("\n2. IT Department Employees:")
print(duckdb.sql("SELECT * FROM read_parquet('employees.parquet') WHERE department = 'IT'").df())

print("\n3. Employees in Delhi:")
print(duckdb.sql("SELECT * FROM read_parquet('employees.parquet') WHERE city = 'Delhi'").df())

print("\n4. IT Employees with Salary > 65,000:")
print(duckdb.sql("SELECT * FROM read_parquet('employees.parquet') WHERE department = 'IT' AND salary > 65000").df())
print("\n" + "="*50 + "\n")

# -------------------------------------------------------------
# Task 4: Select Specific Columns
# -------------------------------------------------------------
print("--- Task 4: Name, Department, Salary Sorted Descending ---")
sorted_cols = duckdb.sql("""
    SELECT name, department, salary
    FROM read_parquet('employees.parquet')
    ORDER BY salary DESC
""").df()
print(sorted_cols)
print("\n" + "="*50 + "\n")

# -------------------------------------------------------------
# Task 5: Perform Aggregations
# -------------------------------------------------------------
print("--- Task 5: Aggregations ---")
summary = duckdb.sql("""
    SELECT
        COUNT(*) AS employee_count,
        AVG(salary) AS average_salary,
        MAX(salary) AS maximum_salary,
        MIN(salary) AS minimum_salary,
        SUM(salary) AS total_salary
    FROM read_parquet('employees.parquet')
""").df()
print(summary)
print("\n" + "="*50 + "\n")

# -------------------------------------------------------------
# Task 6: Group Data by Department
# -------------------------------------------------------------
print("--- Task 6: Group by Department ---")
dept_summary = duckdb.sql("""
    SELECT
        department,
        COUNT(*) AS employee_count,
        AVG(salary) AS average_salary,
        MAX(salary) AS highest_salary,
        SUM(salary) AS total_salary
    FROM read_parquet('employees.parquet')
    GROUP BY department
    ORDER BY average_salary DESC
""").df()
print(dept_summary)
print("\n" + "="*50 + "\n")

# -------------------------------------------------------------
# Task 7: Create a DuckDB Table (company.duckdb)
# -------------------------------------------------------------
print("--- Task 7: Create Persistent Table in company.duckdb ---")
connection = duckdb.connect("company.duckdb")
connection.execute("""
    CREATE OR REPLACE TABLE employees AS
    SELECT * FROM read_parquet('employees.parquet')
""")
db_result = connection.execute("SELECT * FROM employees").df()
print("Table successfully loaded in DuckDB file database:")
print(db_result)
connection.close()
print("\n" + "="*50 + "\n")

# -------------------------------------------------------------
# Task 8: Export Query Results to Parquet
# -------------------------------------------------------------
duckdb.sql("""
    COPY (
        SELECT *
        FROM read_parquet('employees.parquet')
        WHERE salary > 50000
    )
    TO 'high_salary_employees.parquet'
    (FORMAT PARQUET)
""")
duckdb.sql("""
    COPY (
        SELECT *
        FROM read_parquet('employees.parquet')
        WHERE salary > 50000
    )
    TO 'data/high_salary_employees.parquet'
    (FORMAT PARQUET)
""")
print("✅ Task 8: Filtered Parquet file 'high_salary_employees.parquet' created.\n")

# -------------------------------------------------------------
# Task 9: Verify the Exported File
# -------------------------------------------------------------
print("--- Task 9: Verified Exported File Contents ---")
exported_verify = duckdb.sql("SELECT * FROM read_parquet('high_salary_employees.parquet')").df()
print(exported_verify)
print("\n" + "="*50 + "\n")

# -------------------------------------------------------------
# Bonus Tasks
# -------------------------------------------------------------
print("--- Bonus Tasks ---")

# 1. Employee with second-highest salary
second_highest = duckdb.sql("""
    SELECT name, salary
    FROM read_parquet('employees.parquet')
    ORDER BY salary DESC
    LIMIT 1 OFFSET 1
""").df()
print("1. Employee with 2nd Highest Salary:")
print(second_highest)

# 2. Top three highest-paid employees
top_3 = duckdb.sql("""
    SELECT name, department, salary
    FROM read_parquet('employees.parquet')
    ORDER BY salary DESC
    LIMIT 3
""").df()
print("\n2. Top 3 Highest-Paid Employees:")
print(top_3)

# 3. Average salary for each city
city_avg = duckdb.sql("""
    SELECT city, AVG(salary) AS avg_salary
    FROM read_parquet('employees.parquet')
    GROUP BY city
    ORDER BY avg_salary DESC
""").df()
print("\n3. Average Salary by City:")
print(city_avg)

# 4. Departments with average salary > 55,000
high_dept = duckdb.sql("""
    SELECT department, AVG(salary) AS avg_salary
    FROM read_parquet('employees.parquet')
    GROUP BY department
    HAVING AVG(salary) > 55000
""").df()
print("\n4. Departments with Avg Salary > 55,000:")
print(high_dept)

# 5. Salary Category column using CASE statement
salary_cat = duckdb.sql("""
    SELECT
        name,
        salary,
        CASE
            WHEN salary >= 65000 THEN 'High'
            WHEN salary >= 50000 THEN 'Medium'
            ELSE 'Low'
        END AS salary_category
    FROM read_parquet('employees.parquet')
""").df()
print("\n5. Salary Categories:")
print(salary_cat)

# -------------------------------------------------------------
# Generate duckdb_parquet_assignment.py
# -------------------------------------------------------------
script_code = """import pandas as pd
import duckdb

# Task 1: Create Parquet File
data = {
    "employee_id": [1, 2, 3, 4, 5, 6, 7, 8],
    "name": ["Asha", "Rahul", "Neha", "Vikram", "Priya", "Arjun", "Meera", "Karan"],
    "department": ["IT", "HR", "IT", "Finance", "HR", "Finance", "IT", "Sales"],
    "salary": [60000, 45000, 70000, 55000, 48000, 65000, 75000, 50000],
    "city": ["Delhi", "Mumbai", "Bengaluru", "Delhi", "Mumbai", "Chennai", "Bengaluru", "Delhi"]
}
df = pd.DataFrame(data)
df.to_parquet("employees.parquet", index=False)

# Task 2: Read Parquet Using DuckDB
print("All Employees:")
print(duckdb.sql("SELECT * FROM read_parquet('employees.parquet')").df())

# Task 3: Filter Employee Records
print("\\nEmployees with Salary > 50000:")
print(duckdb.sql("SELECT * FROM read_parquet('employees.parquet') WHERE salary > 50000").df())

# Task 4: Select Specific Columns
print("\\nSelected Columns Sorted:")
print(duckdb.sql("SELECT name, department, salary FROM read_parquet('employees.parquet') ORDER BY salary DESC").df())

# Task 5: Aggregations
print("\\nAggregations Summary:")
print(duckdb.sql("SELECT COUNT(*) AS count, AVG(salary) AS avg_sal, MAX(salary) AS max_sal, MIN(salary) AS min_sal, SUM(salary) AS total_sal FROM read_parquet('employees.parquet')").df())

# Task 6: Group Data
print("\\nGroup by Department:")
print(duckdb.sql("SELECT department, COUNT(*) AS count, AVG(salary) AS avg_sal, MAX(salary) AS max_sal, SUM(salary) AS total_sal FROM read_parquet('employees.parquet') GROUP BY department ORDER BY avg_sal DESC").df())

# Task 7: Create DuckDB Table
conn = duckdb.connect("company.duckdb")
conn.execute("CREATE OR REPLACE TABLE employees AS SELECT * FROM read_parquet('employees.parquet')")
conn.close()

# Task 8: Export Query Results
duckdb.sql("COPY (SELECT * FROM read_parquet('employees.parquet') WHERE salary > 50000) TO 'high_salary_employees.parquet' (FORMAT PARQUET)")

# Task 9: Verify Exported File
print("\\nVerified Exported High Salary Employees:")
print(duckdb.sql("SELECT * FROM read_parquet('high_salary_employees.parquet')").df())

# Bonus: Salary Categories
print("\\nSalary Categories:")
print(duckdb.sql(\"\"\"
    SELECT name, salary,
        CASE
            WHEN salary >= 65000 THEN 'High'
            WHEN salary >= 50000 THEN 'Medium'
            ELSE 'Low'
        END AS salary_category
    FROM read_parquet('employees.parquet')
\"\"\").df())
"""

with open('duckdb_parquet_assignment.py', 'w') as f:
    f.write(script_code)

print("\n✅ Saved 'duckdb_parquet_assignment.py' successfully!")

✅ Task 1: 'employees.parquet' created successfully.

--- Task 2: All Employee Records ---
   employee_id    name department  salary       city
0            1    Asha         IT   60000      Delhi
1            2   Rahul         HR   45000     Mumbai
2            3    Neha         IT   70000  Bengaluru
3            4  Vikram    Finance   55000      Delhi
4            5   Priya         HR   48000     Mumbai
5            6   Arjun    Finance   65000    Chennai
6            7   Meera         IT   75000  Bengaluru
7            8   Karan      Sales   50000      Delhi


--- Task 3: Filtered Queries ---
1. Salary > 50,000:
   employee_id    name department  salary       city
0            1    Asha         IT   60000      Delhi
1            3    Neha         IT   70000  Bengaluru
2            4  Vikram    Finance   55000      Delhi
3            6   Arjun    Finance   65000    Chennai
4            7   Meera         IT   75000  Bengaluru

2. IT Department Employees:
   employee_id   name departmen

In [3]:
import os
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
import pyarrow.ipc as ipc
import pandas as pd

# Create required directory
os.makedirs('data', exist_ok=True)

print("="*60)
print("TASK 12: APACHE ARROW ASSIGNMENT")
print("="*60)

# -------------------------------------------------------------
# Task 1: Create an Arrow Table
# -------------------------------------------------------------
data = {
    "employee_id": [1, 2, 3, 4, 5, 6],
    "name": ["Asha", "Rahul", "Neha", "Vikram", "Priya", "Arjun"],
    "department": ["IT", "HR", "IT", "Finance", "HR", "Finance"],
    "salary": [60000, 45000, 70000, 55000, 48000, 65000],
    "city": ["Delhi", "Mumbai", "Bengaluru", "Delhi", "Mumbai", "Chennai"]
}

employee_table = pa.table(data)
print("\n--- Task 1: Employee Arrow Table ---")
print(employee_table)

# -------------------------------------------------------------
# Task 2: Display the Schema
# -------------------------------------------------------------
print("\n--- Task 2: Arrow Table Schema ---")
print(employee_table.schema)
print("\nData Types:")
print("  - employee_id:", employee_table.schema.field("employee_id").type)
print("  - name:", employee_table.schema.field("name").type)
print("  - salary:", employee_table.schema.field("salary").type)

# -------------------------------------------------------------
# Task 3: Inspect the Table
# -------------------------------------------------------------
print("\n--- Task 3: Table Inspection ---")
print("Rows:", employee_table.num_rows)
print("Columns:", employee_table.num_columns)
print("Column names:", employee_table.column_names)
print("\n'name' Column:")
print(employee_table.column("name"))
print("\nFirst 3 Rows:")
print(employee_table.slice(0, 3))

# -------------------------------------------------------------
# Task 4: Select Specific Columns
# -------------------------------------------------------------
selected_table = employee_table.select(["name", "department", "salary"])
print("\n--- Task 4: Selected Columns (Name, Department, Salary) ---")
print(selected_table)

# -------------------------------------------------------------
# Task 5: Filter Records (Salary > 50,000)
# -------------------------------------------------------------
salary_filter = pc.greater(employee_table["salary"], 50000)
high_salary_table = employee_table.filter(salary_filter)
print("\n--- Task 5: High Salary Employees (>50,000) ---")
print(high_salary_table)

# -------------------------------------------------------------
# Task 6: Filter by Department (IT)
# -------------------------------------------------------------
department_filter = pc.equal(employee_table["department"], "IT")
it_employees = employee_table.filter(department_filter)
print("\n--- Task 6: IT Department Employees ---")
print(it_employees)

# -------------------------------------------------------------
# Task 7: Perform Calculations
# -------------------------------------------------------------
salary_column = employee_table["salary"]
print("\n--- Task 7: Calculations ---")
print("Average salary:", pc.mean(salary_column).as_py())
print("Maximum salary:", pc.max(salary_column).as_py())
print("Minimum salary:", pc.min(salary_column).as_py())
print("Total salary:", pc.sum(salary_column).as_py())

# -------------------------------------------------------------
# Task 8: Add a New Column (bonus = 10%)
# -------------------------------------------------------------
bonus_column = pc.multiply(employee_table["salary"], 0.10)
employee_table = employee_table.append_column("bonus", bonus_column)
print("\n--- Task 8: Table with Added 'bonus' Column ---")
print(employee_table)

# -------------------------------------------------------------
# Task 9: Convert Arrow to Pandas
# -------------------------------------------------------------
employee_df = employee_table.to_pandas()
print("\n--- Task 9: Pandas DataFrame ---")
print(employee_df)

# -------------------------------------------------------------
# Task 10: Convert Pandas to Arrow
# -------------------------------------------------------------
new_arrow_table = pa.Table.from_pandas(employee_df, preserve_index=False)
print("\n--- Task 10: Converted Back to Arrow Table ---")
print(new_arrow_table)

# -------------------------------------------------------------
# Task 11: Save as Parquet File
# -------------------------------------------------------------
pq.write_table(employee_table, "employees.parquet")
pq.write_table(employee_table, "data/employees.parquet")
print("\n✅ Task 11: Saved 'employees.parquet'")

# -------------------------------------------------------------
# Task 12: Read the Parquet File
# -------------------------------------------------------------
loaded_parquet_table = pq.read_table("employees.parquet")
print("\n--- Task 12: Loaded Parquet File ---")
print(loaded_parquet_table)

# -------------------------------------------------------------
# Task 13: Save as Arrow IPC File
# -------------------------------------------------------------
with ipc.new_file("employees.arrow", employee_table.schema) as writer:
    writer.write_table(employee_table)

with ipc.new_file("data/employees.arrow", employee_table.schema) as writer:
    writer.write_table(employee_table)

print("\n✅ Task 13: Saved 'employees.arrow' (IPC Format)")

# -------------------------------------------------------------
# Task 14: Read the Arrow IPC File
# -------------------------------------------------------------
with ipc.open_file("employees.arrow") as reader:
    ipc_table = reader.read_all()
print("\n--- Task 14: Read Arrow IPC File ---")
print(ipc_table)

# -------------------------------------------------------------
# Bonus Tasks
# -------------------------------------------------------------
print("\n--- Bonus Tasks ---")

# Bonus 1: Employees in Delhi
delhi_filter = pc.equal(employee_table["city"], "Delhi")
delhi_employees = employee_table.filter(delhi_filter)
print("\n1. Employees in Delhi:")
print(delhi_employees)

# Bonus 2: Salary between 50,000 and 65,000
range_filter = pc.and_(
    pc.greater_equal(employee_table["salary"], 50000),
    pc.less_equal(employee_table["salary"], 65000)
)
salary_range_table = employee_table.filter(range_filter)
print("\n2. Employees with Salary Between 50k and 65k:")
print(salary_range_table)

# Bonus 3: Add annual_salary column (salary * 12)
annual_salary_col = pc.multiply(employee_table["salary"], 12)
employee_table_annual = employee_table.append_column("annual_salary", annual_salary_col)
print("\n3. Table with 'annual_salary' Column:")
print(employee_table_annual)

# Bonus 4: Save IT employees to it_employees.parquet
pq.write_table(it_employees, "it_employees.parquet")
print("\n4. ✅ Saved IT employees to 'it_employees.parquet'")

# Bonus 5: Read only 'name' and 'salary' columns from Parquet file
pruned_pq = pq.read_table("employees.parquet", columns=["name", "salary"])
print("\n5. Column Pruned Read ('name' & 'salary' only):")
print(pruned_pq)

# Bonus 6: Sort employees by salary from highest to lowest (FIXED)
indices = pc.sort_indices(employee_table, sort_keys=[("salary", "descending")])
sorted_table = employee_table.take(indices)
print("\n6. Employees Sorted by Salary Descending:")
print(sorted_table)


# -------------------------------------------------------------
# Generate standalone apache_arrow_assignment.py
# -------------------------------------------------------------
script_code = """import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
import pyarrow.ipc as ipc
import pandas as pd

# Task 1: Create Arrow Table
data = {
    "employee_id": [1, 2, 3, 4, 5, 6],
    "name": ["Asha", "Rahul", "Neha", "Vikram", "Priya", "Arjun"],
    "department": ["IT", "HR", "IT", "Finance", "HR", "Finance"],
    "salary": [60000, 45000, 70000, 55000, 48000, 65000],
    "city": ["Delhi", "Mumbai", "Bengaluru", "Delhi", "Mumbai", "Chennai"]
}
employee_table = pa.table(data)
print("Task 1 - Arrow Table created:")
print(employee_table)

# Task 2: Display Schema
print("\\nTask 2 - Schema:")
print(employee_table.schema)

# Task 3: Inspect Table
print("\\nTask 3 - Inspecting:")
print("Rows:", employee_table.num_rows)
print("Columns:", employee_table.num_columns)

# Task 4: Select Specific Columns
print("\\nTask 4 - Selected Columns:")
print(employee_table.select(["name", "department", "salary"]))

# Task 5: Filter Salary > 50000
print("\\nTask 5 - High Salary:")
print(employee_table.filter(pc.greater(employee_table["salary"], 50000)))

# Task 6: Filter IT Department
print("\\nTask 6 - IT Department:")
it_employees = employee_table.filter(pc.equal(employee_table["department"], "IT"))
print(it_employees)

# Task 7: Calculations
sal = employee_table["salary"]
print(f"\\nTask 7 - Calculations: Avg={pc.mean(sal).as_py()}, Max={pc.max(sal).as_py()}, Min={pc.min(sal).as_py()}, Sum={pc.sum(sal).as_py()}")

# Task 8: Add Bonus Column
bonus_col = pc.multiply(employee_table["salary"], 0.10)
employee_table = employee_table.append_column("bonus", bonus_col)

# Task 9 & 10: Convert Arrow <-> Pandas
df = employee_table.to_pandas()
new_table = pa.Table.from_pandas(df, preserve_index=False)

# Task 11 & 12: Parquet Read/Write
pq.write_table(employee_table, "employees.parquet")
print("\\nTask 12 - Loaded Parquet:")
print(pq.read_table("employees.parquet"))

# Task 13 & 14: Arrow IPC Read/Write
with ipc.new_file("employees.arrow", employee_table.schema) as writer:
    writer.write_table(employee_table)

with ipc.open_file("employees.arrow") as reader:
    ipc_table = reader.read_all()
print("\\nTask 14 - Loaded IPC:")
print(ipc_table)

# Bonus Tasks
pq.write_table(it_employees, "it_employees.parquet")
"""

with open('apache_arrow_assignment.py', 'w') as f:
    f.write(script_code)

print("\n✅ Saved 'apache_arrow_assignment.py' successfully!")

TASK 12: APACHE ARROW ASSIGNMENT

--- Task 1: Employee Arrow Table ---
pyarrow.Table
employee_id: int64
name: string
department: string
salary: int64
city: string
----
employee_id: [[1,2,3,4,5,6]]
name: [["Asha","Rahul","Neha","Vikram","Priya","Arjun"]]
department: [["IT","HR","IT","Finance","HR","Finance"]]
salary: [[60000,45000,70000,55000,48000,65000]]
city: [["Delhi","Mumbai","Bengaluru","Delhi","Mumbai","Chennai"]]

--- Task 2: Arrow Table Schema ---
employee_id: int64
name: string
department: string
salary: int64
city: string

Data Types:
  - employee_id: int64
  - name: string
  - salary: int64

--- Task 3: Table Inspection ---
Rows: 6
Columns: 5
Column names: ['employee_id', 'name', 'department', 'salary', 'city']

'name' Column:
[
  [
    "Asha",
    "Rahul",
    "Neha",
    "Vikram",
    "Priya",
    "Arjun"
  ]
]

First 3 Rows:
pyarrow.Table
employee_id: int64
name: string
department: string
salary: int64
city: string
----
employee_id: [[1,2,3]]
name: [["Asha","Rahul","Neha"